# zero_shot_best_candidates_full_run

Runs both generative and non-generative zero-shot ablations in one notebook.

In [1]:
from pathlib import Path
from itertools import product
import sys
import json
import pandas as pd
import matplotlib.pyplot as plt

import torch
import transformers
from transformers import AutoProcessor


In [2]:
try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    drive.mount('/content/drive')
    CANDIDATE_ROOT = Path('/content/drive/MyDrive/DL_Final_Project')
    PROJECT_ROOT = CANDIDATE_ROOT if CANDIDATE_ROOT.exists() else Path.cwd()
else:
    cwd = Path.cwd().resolve()
    if cwd.name == 'notebooks':
        PROJECT_ROOT = cwd.parent
    elif cwd.name == 'zero_shot_study' and cwd.parent.name == 'notebooks':
        PROJECT_ROOT = cwd.parent.parent
    else:
        PROJECT_ROOT = cwd

print('IN_COLAB:', IN_COLAB)
print('PROJECT_ROOT:', PROJECT_ROOT)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.zero_shot_ablation_utils import (
    set_seed,
    load_split,
    apply_sanity_subset,
    cap_validation_rows,
    select_ablation_configs,
    run_generative_ablation,
    summarize_generative_predictions,
    run_nongenerative_ablation,
    summarize_nongenerative_predictions,
    top_configs,
    make_submission,
    validate_submission,
    build_failure_examples,
    maybe_save_dataframe,
    maybe_save_json,
    maybe_save_figure,
)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
IN_COLAB: True
PROJECT_ROOT: /content/drive/MyDrive/DL_Final_Project


In [3]:
save = True #False
generate_submission = True #False
sanity_check = False #True
n = 32
seed = 42
max_val_examples = None
top_k = 5
batch_size = 16
load_saved_results = True

DATA_DIR = PROJECT_ROOT / 'data'
IMAGE_DATA_DIR = DATA_DIR / 'images'
OUT_DIR = PROJECT_ROOT / 'outputs' / 'zero_shot_best_candidates_full_run'

MODEL_ID = 'HuggingFaceTB/SmolVLM-500M-Instruct'
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

set_seed(seed)
if save:
    OUT_DIR.mkdir(parents=True, exist_ok=True)


In [4]:
print('torch version:', torch.__version__)
print('cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('cuda device name:', torch.cuda.get_device_name(torch.cuda.current_device()))


torch version: 2.10.0+cu128
cuda available: True
cuda device name: NVIDIA A100-SXM4-40GB


In [5]:
train_df = load_split(DATA_DIR, 'train')
val_df = load_split(DATA_DIR, 'val')
test_df = load_split(DATA_DIR, 'test')

train_df = apply_sanity_subset(train_df, sanity_check=sanity_check, n=n, seed=seed)
val_df = apply_sanity_subset(val_df, sanity_check=sanity_check, n=n, seed=seed)
test_df = apply_sanity_subset(test_df, sanity_check=sanity_check, n=n, seed=seed)
val_df = cap_validation_rows(val_df, max_val_examples=max_val_examples)

print('train rows used:', len(train_df))
print('validation rows used:', len(val_df))
print('test rows used:', len(test_df))


train rows used: 3109
validation rows used: 1048
test rows used: 1008


In [6]:
print('transformers version:', transformers.__version__)
if hasattr(transformers, 'AutoModelForVision2Seq'):
    ModelClass = transformers.AutoModelForVision2Seq
elif hasattr(transformers, 'AutoModelForImageTextToText'):
    ModelClass = transformers.AutoModelForImageTextToText
else:
    raise ImportError('Upgrade transformers: missing SmolVLM model class.')

processor = AutoProcessor.from_pretrained(MODEL_ID)
# Batched generation with decoder-only models should left-pad.
if hasattr(processor, 'tokenizer') and processor.tokenizer is not None:
    processor.tokenizer.padding_side = 'left'
    if processor.tokenizer.pad_token is None and processor.tokenizer.eos_token is not None:
        processor.tokenizer.pad_token = processor.tokenizer.eos_token

model = ModelClass.from_pretrained(MODEL_ID).to(DEVICE)
model.eval()


transformers version: 5.0.0


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/489 [00:00<?, ?it/s]

Idefics3ForConditionalGeneration(
  (model): Idefics3Model(
    (vision_model): Idefics3VisionTransformer(
      (embeddings): Idefics3VisionEmbeddings(
        (patch_embedding): Conv2d(3, 768, kernel_size=(16, 16), stride=(16, 16), padding=valid)
        (position_embedding): Embedding(1024, 768)
      )
      (encoder): Idefics3Encoder(
        (layers): ModuleList(
          (0-11): 12 x Idefics3EncoderLayer(
            (self_attn): Idefics3VisionAttention(
              (k_proj): Linear(in_features=768, out_features=768, bias=True)
              (v_proj): Linear(in_features=768, out_features=768, bias=True)
              (q_proj): Linear(in_features=768, out_features=768, bias=True)
              (out_proj): Linear(in_features=768, out_features=768, bias=True)
            )
            (layer_norm1): LayerNorm((768,), eps=1e-06, elementwise_affine=True)
            (mlp): Idefics3VisionMLP(
              (activation_fn): GELUTanh()
              (fc1): Linear(in_features=768, out

## Candidate Spaces and Selection

In [7]:
# Define ablations directly by name.
# Each key is an ablation name. Value can be:
#   - a params dictionary (single ablation), or
#   - a list of params dictionaries (multiple variants under one name).
# Empty dictionary means skip that family entirely.

generative_ablation_space = {
    # block3_016 — best validation acc (~66%): greedy edged beam

    # block3_004 / Block 2 lineage — hint_only + answer contract (~60% class)
    'ablation4': {
        'prompt_structure': 'question_first',
        'context_mode': 'hint_only',
        'choice_format': 'letter_dot',
        'output_format': 'answer_prefix',
        'max_new_tokens': 3072,
        'decoding_strategy': 'greedy',
        'parse_rule': 'answer_prefix',
    },
    'ablation5': {
        'prompt_structure': 'context_first',
        'context_mode': 'hint_lecture',
        'choice_format': 'letter_dot',
        'output_format': 'answer_prefix',
        'max_new_tokens': 3072,
        'decoding_strategy': 'greedy',
        'parse_rule': 'answer_prefix',
    },



}



nongenerative_ablation_space = {}

def build_configs_from_named_space(named_space: dict, prefix: str) -> list[dict]:
    configs = []
    idx = 1
    for name, spec in named_space.items():
        variants = spec if isinstance(spec, list) else [spec]
        for variant_idx, params in enumerate(variants, start=1):
            cfg = dict(params)
            cfg['ablation_name'] = name
            if len(variants) > 1:
                cfg['ablation_id'] = f"{prefix}_{name}_{variant_idx:02d}"
            else:
                cfg['ablation_id'] = f"{prefix}_{name}"
            configs.append(cfg)
            idx += 1
    return configs

all_gen_configs = build_configs_from_named_space(generative_ablation_space, 'gen') if generative_ablation_space else []
all_ng_configs = build_configs_from_named_space(nongenerative_ablation_space, 'nongen') if nongenerative_ablation_space else []

if len(all_gen_configs):
    display(pd.DataFrame(all_gen_configs))
else:
    print('Generative ablation space is empty. Generative approach will be skipped.')

if len(all_ng_configs):
    display(pd.DataFrame(all_ng_configs))
else:
    print('Non-generative ablation space is empty. Non-generative approach will be skipped.')

gen_configs_to_run = list(all_gen_configs)
ng_configs_to_run = list(all_ng_configs)

print('generative configs to run:', len(gen_configs_to_run))
print('non-generative configs to run:', len(ng_configs_to_run))


,prompt_structure,context_mode,choice_format,output_format,max_new_tokens,decoding_strategy,parse_rule,ablation_name,ablation_id
0,question_first,hint_only,letter_dot,answer_prefix,3072,greedy,answer_prefix,ablation4,gen_ablation4
1,context_first,hint_lecture,letter_dot,answer_prefix,3072,greedy,answer_prefix,ablation5,gen_ablation5


Non-generative ablation space is empty. Non-generative approach will be skipped.
generative configs to run: 2
non-generative configs to run: 0


In [8]:
gen_results, gen_preds = [], []
if len(gen_configs_to_run):
    for config in gen_configs_to_run:
        val_pred_df = run_generative_ablation(
            val_df,
            IMAGE_DATA_DIR,
            processor,
            model,
            DEVICE,
            config,
            batch_size=batch_size,
        )
        gen_preds.append(val_pred_df)
        gen_results.append(summarize_generative_predictions(val_pred_df, config))
else:
    print('Skipping generative runs (generative_ablation_space is empty).')

ng_results, ng_preds = [], []
if len(ng_configs_to_run):
    for config in ng_configs_to_run:
        val_pred_df = run_nongenerative_ablation(
            val_df,
            IMAGE_DATA_DIR,
            processor,
            model,
            DEVICE,
            config,
            batch_size=batch_size,
        )
        ng_preds.append(val_pred_df)
        ng_results.append(summarize_nongenerative_predictions(val_pred_df, config))
else:
    print('Skipping non-generative runs (nongenerative_ablation_space is empty).')

gen_results_df = pd.DataFrame(gen_results)
if len(gen_results_df):
    gen_results_df = gen_results_df.sort_values('accuracy', ascending=False).reset_index(drop=True)

ng_results_df = pd.DataFrame(ng_results)
if len(ng_results_df):
    ng_results_df = ng_results_df.sort_values('accuracy', ascending=False).reset_index(drop=True)

gen_predictions_df = pd.concat(gen_preds, ignore_index=True) if gen_preds else pd.DataFrame()
ng_predictions_df = pd.concat(ng_preds, ignore_index=True) if ng_preds else pd.DataFrame()


Gen gen_ablation4:   0%|          | 0/66 [00:00<?, ?it/s]

Gen gen_ablation5:   0%|          | 0/66 [00:00<?, ?it/s]

Skipping non-generative runs (nongenerative_ablation_space is empty).


In [9]:
if len(gen_results_df):
    display(gen_results_df[['ablation_id', 'percent_correct', 'accuracy', 'parse_failure_rate', 'max_new_tokens']])
else:
    print('No generative results to display.')

if len(ng_results_df):
    display(ng_results_df[['ablation_id', 'percent_correct', 'accuracy', 'mean_confidence_margin', 'scoring_target']])
else:
    print('No non-generative results to display.')


,ablation_id,percent_correct,accuracy,parse_failure_rate,max_new_tokens
0,gen_ablation4,62.022901,0.620229,0.035305,3072
1,gen_ablation5,61.450382,0.614504,0.048664,3072


No non-generative results to display.


In [10]:
gen_top_cfgs = top_configs(gen_results_df, k=top_k) if len(gen_results_df) else []
ng_top_cfgs = top_configs(ng_results_df, k=top_k) if len(ng_results_df) else []

if len(gen_results_df):
    maybe_save_dataframe(gen_results_df, OUT_DIR / 'generative_results.csv', save=save)
    maybe_save_dataframe(gen_predictions_df, OUT_DIR / 'generative_predictions.csv', save=save)
    maybe_save_dataframe(build_failure_examples(gen_predictions_df, max_examples=200), OUT_DIR / 'generative_failureexamples.csv', save=save)

if len(ng_results_df):
    maybe_save_dataframe(ng_results_df, OUT_DIR / 'nongenerative_results.csv', save=save)
    maybe_save_dataframe(ng_predictions_df, OUT_DIR / 'nongenerative_predictions.csv', save=save)
    maybe_save_dataframe(build_failure_examples(ng_predictions_df, max_examples=200), OUT_DIR / 'nongenerative_failureexamples.csv', save=save)

maybe_save_json({'generative_top_configs': gen_top_cfgs, 'nongenerative_top_configs': ng_top_cfgs}, OUT_DIR / 'topconfigs.json', save=save)


PosixPath('/content/drive/MyDrive/DL_Final_Project/outputs/zero_shot_best_candidates_full_run/topconfigs.json')

In [ ]:
gen_submission_df, ng_submission_df = None, None
if generate_submission and len(gen_top_cfgs):
    gen_test_pred_df = run_generative_ablation(
        test_df,
        IMAGE_DATA_DIR,
        processor,
        model,
        DEVICE,
        gen_top_cfgs[0],
        batch_size=batch_size,
    )
    gen_submission_df = make_submission(gen_test_pred_df)
    validate_submission(gen_submission_df)
    maybe_save_dataframe(gen_test_pred_df, OUT_DIR / 'generative_test_predictions.csv', save=save)
    maybe_save_dataframe(gen_submission_df, OUT_DIR / 'generative_submission.csv', save=save)

if generate_submission and len(ng_top_cfgs):
    ng_test_pred_df = run_nongenerative_ablation(
        test_df,
        IMAGE_DATA_DIR,
        processor,
        model,
        DEVICE,
        ng_top_cfgs[0],
        batch_size=batch_size,
    )
    ng_submission_df = make_submission(ng_test_pred_df)
    validate_submission(ng_submission_df)
    maybe_save_dataframe(ng_test_pred_df, OUT_DIR / 'nongenerative_test_predictions.csv', save=save)
    maybe_save_dataframe(ng_submission_df, OUT_DIR / 'nongenerative_submission.csv', save=save)


Gen gen_ablation4:   0%|          | 0/63 [00:00<?, ?it/s]

In [ ]:
if not sanity_check and generate_submission:
    expected_rows = len(load_split(DATA_DIR, 'test'))
    if gen_submission_df is not None:
        assert len(gen_submission_df) == expected_rows
    if ng_submission_df is not None:
        assert len(ng_submission_df) == expected_rows
    print('Full-test row count checks passed.')
else:
    print('Skipping full-test safety checks.')


Full-test row count checks passed.
